# Baseline SB3 DQN — Highway-v0

Ce notebook présente les résultats de la baseline DQN entraînée avec Stable-Baselines3 sur le **core task** du projet RL.

La configuration du benchmark est importée depuis `shared_core_config.py` et n'est pas modifiée.

**Protocole d'évaluation :** politique déterministe, 50 épisodes, seeds fixes (`EVAL_SEEDS` dans `utils.py`).

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Video, display  # noqa: A004

HERE = Path.cwd()
if HERE.name == "src":
    os.chdir(HERE.parent)
    HERE = HERE.parent

sys.path.insert(0, str(HERE / "src"))
import shared_core_config  # noqa: E402

RESULTS_DIR = HERE / "results" / "sb3_dqn"
VIDEOS_DIR = HERE / "videos"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

seed_dirs = sorted(RESULTS_DIR.glob("seed_*")) if RESULTS_DIR.exists() else []
print(f"Seeds trouvées : {[d.name for d in seed_dirs] or 'aucune — lancez train_sb3_dqn.py'}")

## Config du benchmark

Config importée depuis `shared_core_config.py` — source unique de vérité pour tous les entraînements.

In [ ]:
print(f"Environnement : {shared_core_config.SHARED_CORE_ENV_ID}\n")
for key, val in shared_core_config.SHARED_CORE_CONFIG.items():
    print(f"  {key}: {val}")

## Hyperparamètres DQN

**Choix : DQN avec MlpPolicy**
- Espace d'action : discret (5 actions `DiscreteMetaAction`) → DQN natif
- Espace d'observation : continu (10 véhicules × 5 features = 50 dim, aplati par `FlattenExtractor`) → MLP adapté

| Hyperparamètre | Valeur de départ | Rôle | Valeurs à tester |
|---|---|---|---|
| `learning_rate` | 5e-4 | Vitesse d'apprentissage | 1e-4, 1e-3 |
| `buffer_size` | 50 000 | Taille replay buffer | 20k, 100k |
| `batch_size` | 64 | Taille mini-batch | 32, 128 |
| `gamma` | 0.99 | Discount futur | 0.95, 0.999 |
| `exploration_fraction` | 0.15 | % steps pour décroître epsilon | 0.1, 0.3 |
| `exploration_final_eps` | 0.05 | Epsilon final | 0.01, 0.1 |
| `target_update_interval` | 1 000 | Fréquence mise à jour target net | 500, 2000 |
| `net_arch` | [256, 256] | Architecture MLP | [128,128], [256,256,256] |

In [ ]:
if seed_dirs:
    with open(seed_dirs[0] / "hparams.json") as f:
        hparams = json.load(f)
    print("Hyperparamètres utilisés pour l'entraînement :\n")
    for k, v in hparams.items():
        print(f"  {k}: {v}")
else:
    print("Aucun résultat. Lancez : uv run python scripts/train_sb3_dqn.py --seed 0")

## Courbes d'entraînement

- **Gauche** : reward par épisode (lissé, fenêtre=20) depuis les logs Monitor
- **Droite** : reward d'évaluation périodique (EvalCallback, tous les 10 000 steps, moyenne sur 10 épisodes)

In [ ]:
def load_monitor(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, skiprows=1)


def smooth(values: np.ndarray, window: int = 20) -> np.ndarray:
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="same")


if not seed_dirs:
    print("Aucun résultat. Lancez scripts/train_sb3_dqn.py")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for seed_dir in seed_dirs:
        label = seed_dir.name
        monitor_path = seed_dir / "monitor.monitor.csv"
        eval_path = seed_dir / "eval_logs" / "evaluations.npz"

        if monitor_path.exists():
            df = load_monitor(monitor_path)
            smoothed = smooth(df["r"].values)
            axes[0].plot(range(len(smoothed)), smoothed, alpha=0.8, label=label)

        if eval_path.exists():
            data = np.load(eval_path)
            ts = data["timesteps"]
            mean = data["results"].mean(axis=1)
            std = data["results"].std(axis=1)
            axes[1].plot(ts, mean, marker="o", markersize=3, label=label)
            axes[1].fill_between(ts, mean - std, mean + std, alpha=0.15)

    axes[0].set_title("Reward d'entraînement (lissé, fenêtre=20)")
    axes[0].set_xlabel("Épisode")
    axes[0].set_ylabel("Reward")
    axes[0].legend()

    axes[1].set_title("Reward d'évaluation périodique")
    axes[1].set_xlabel("Timesteps")
    axes[1].set_ylabel("Mean reward (10 épisodes)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Résultats d'évaluation finale

Évaluation sur **50 épisodes**, politique déterministe, seeds fixes.
Lancer pour chaque seed :
```bash
uv run python scripts/evaluate_sb3_dqn.py --model-path results/sb3_dqn/seed_0/best_model/best_model.zip
```

In [ ]:
results = []
for seed_dir in seed_dirs:
    for candidate in [
        seed_dir / "eval_results.json",
        seed_dir / "best_model" / "eval_results.json",
    ]:
        if candidate.exists():
            with open(candidate) as f:
                results.append(json.load(f))
            break

if results:
    df_res = pd.DataFrame(results)
    cols = [
        "seed",
        "mean_reward",
        "std_reward",
        "crash_rate",
        "offroad_rate",
        "mean_speed",
        "mean_episode_length",
    ]
    cols = [c for c in cols if c in df_res.columns]
    display(df_res[cols].round(3))
else:
    print("Aucun résultat d'évaluation. Lancez scripts/evaluate_sb3_dqn.py")

In [ ]:
if results:
    keys = ["mean_reward", "crash_rate", "offroad_rate", "mean_speed", "mean_episode_length"]
    summary = {}
    for k in keys:
        if k in results[0]:
            vals = [r[k] for r in results]
            summary[k] = f"{np.mean(vals):.3f} ± {np.std(vals):.3f}"

    print("Tableau récapitulatif (moyenne ± std sur toutes les seeds) :\n")
    for k, v in summary.items():
        print(f"  {k:25s}: {v}")

## Visualisation qualitative

Générer les vidéos avec :
```bash
uv run python scripts/record_rollout.py --model-path results/sb3_dqn/seed_0/best_model/best_model.zip
```
Les vidéos sont sauvegardées dans `videos/<model_stem>/seed<N>/`.

In [ ]:
video_files = sorted(VIDEOS_DIR.glob("**/*.mp4")) if VIDEOS_DIR.exists() else []

if video_files:
    print(f"{len(video_files)} vidéo(s) trouvée(s). Affichage : {video_files[0].name}\n")
    display(Video(str(video_files[0]), embed=True, width=640))
else:
    print("Aucune vidéo. Lancez scripts/record_rollout.py pour en générer.")

## Interprétation

Points à analyser après entraînement :

- **Convergence** : la courbe d'évaluation doit se stabiliser. Une courbe qui oscille fortement indique un buffer trop petit ou un learning rate trop élevé.
- **Crash rate** : doit diminuer au fil de l'entraînement. Un taux > 0.5 en fin d'entraînement suggère que l'agent n'a pas appris à éviter les collisions.
- **Vitesse moyenne** : l'agent bien entraîné doit rouler dans la plage [22, 30] m/s (récompensée par `high_speed_reward`).
- **Variance inter-seeds** : une forte variance sur `mean_reward` indique une sensibilité à l'initialisation — il faudra moyenner sur plus de seeds.

Seuils attendus pour une bonne baseline :
- Mean reward > 15, crash rate < 0.30, vitesse moyenne > 22 m/s

## Limites et failure cases

- **Observation partielle** : seulement 10 véhicules observés sur 45 — l'agent a une vue limitée de l'environnement dense.
- **Replay buffer uniforme** : pas de prioritization (PER) — les transitions rares (collisions) sont sous-représentées.
- **Exploration epsilon-greedy** : peut rater des comportements complexes (double changement de voie, dépassement).
- **Sensibilité aux seeds** : les résultats peuvent varier significativement selon la seed d'initialisation des poids.
- **Durée épisode fixe** : 30 secondes — l'agent peut apprendre à survivre passivement sans chercher à aller vite.

## Comparaison future avec le DQN maison

Cette baseline sert de référence. Utiliser les mêmes `EVAL_SEEDS` (définis dans `utils.py`) pour une comparaison équitable.

| Critère | SB3 DQN (baseline) | DQN maison |
|---|---|---|
| Architecture | MLP [256, 256] | À définir |
| Replay buffer | Uniforme, 50k | À définir |
| Target network | Hard update (tau=1.0, interval=1000) | À définir |
| Exploration | Epsilon-greedy linéaire | À définir |
| Mean reward | À compléter | À compléter |
| Crash rate | À compléter | À compléter |
| Mean speed | À compléter | À compléter |